In [ ]:
%pip install llama_stack_client==0.2.22 fire docling

In [ ]:
from llama_stack_client import RAGDocument, LlamaStackClient
from docling.document_converter import DocumentConverter

In [ ]:
client = LlamaStackClient(base_url="http://llamastack-with-config-service.llama-stack.svc.cluster.local:8321")

In [ ]:
models = client.models.list()

In [ ]:
model_id = next(m.identifier for m in models if m.model_type == "llm")

embedding_model = next(m for m in models if m.model_type == "embedding")
embedding_model_id = embedding_model.identifier
embedding_dimension = int(embedding_model.metadata["embedding_dimension"])

In [ ]:
vector_db_id = "my_pgvector_db"
actual_vector_db_id = vector_store.id

In [ ]:
# RAW Text Ingestion
raw_text = """
LlamaStack can embed raw text into a vector store for retrieval.
This example ingests a small passage for demonstration.
"""
document = RAGDocument(
    document_id="raw_text_001",
    content=raw_text,
    mime_type="text/plain",
    metadata={"source": "example_passage"},
)
client.tool_runtime.rag_tool.insert(
    documents=[document],
    vector_db_id=actual_vector_db_id,
    chunk_size_in_tokens=100,
)
print("Raw text ingested successfully")

In [ ]:
# HTML Ingestion
source = "https://www.paulgraham.com/greatwork.html"
document = RAGDocument(
    document_id="document_1",
    content=source,
    mime_type="text/html",
    metadata={},
)
client.tool_runtime.rag_tool.insert(
    documents=[document],
    vector_db_id=actual_vector_db_id,
    chunk_size_in_tokens=50,
)
print("HTML ingested successfully")

In [ ]:
# PDF as Markdown Ingestion
# 1. Initialize the converter
source = "./red_hat_openshift_ai_self-managed-2.16-getting_started_with_red_hat_openshift_ai_self-managed-en-us.pdf"
converter = DocumentConverter()

# 2. Convert the PDF
# This step handles layout analysis, table extraction, etc.
result = converter.convert(source)

# 3. Export to Markdown for the best RAG performance
markdown_content = result.document.export_to_markdown()

# 4. Ingest into your RAGDocument
document = RAGDocument(
    document_id="docling_doc_1",
    content=markdown_content,
    mime_type="text/markdown",  # Note: Use markdown for better LLM reasoning
    metadata={
        "source": source,
        "page_count": len(result.document.pages)
    },
)
client.tool_runtime.rag_tool.insert(
    documents=[document],
    vector_db_id=actual_vector_db_id,
    chunk_size_in_tokens=400,
)
print("PDF as Markdown ingested successfully")